# 01 - Dataset analysis (Phase 2)
Goal: understand the **real** structure of MACD, HateXplain and IndicGuard before any preprocessing.

Run top to bottom. Sections:
1. Setup  2. Raw file structure  3. Load datasets  4. Dataset sizes  5. Class distribution
6. Language distribution  7. Text length  8. Missing values & duplicates  9. IndicGuard hate-content exploration  10. Save summaries

## 1. Setup

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().resolve()
while not (ROOT / "src").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
print("Project root:", ROOT)

import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 160)
sns.set_theme(style="whitegrid")

from src.utils.config import (MACD_DIR, HATEXPLAIN_DIR, INDICGUARD_DIR,
                              FIGURES_DIR, METRICS_DIR, LABEL_NAMES)
from src.data.io_utils import find_data_files, read_any, inspect_frame
from src.data.load_macd import load_macd
from src.data.load_hatexplain import load_hatexplain
from src.data.load_indicguard import load_indicguard
FIGURES_DIR.mkdir(parents=True, exist_ok=True); METRICS_DIR.mkdir(parents=True, exist_ok=True)

## 2. Raw file structure (no assumptions)
For every raw file: shape, columns, dtypes, nulls, duplicates and a few rows.
**Read this output carefully** - it tells you the true column names. If the automatic detection in
`src/utils/config.py` picks the wrong column, set the `*_COL` override there.

In [ ]:
for name, root in [("MACD", MACD_DIR), ("HateXplain", HATEXPLAIN_DIR), ("IndicGuard", INDICGUARD_DIR)]:
    files = find_data_files(root)
    if name == "MACD":        # the other MACD folders are re-splits of the same comments
        files = [f for f in files if (root / "dataset") in f.parents]
    if name == "IndicGuard":  # all languages share one structure; one language is enough here
        files = [f for f in files if f.parent.name == "Hindi"]
    print("#" * 90); print(f"{name}: inspecting {len(files)} data file(s) in {root}")
    for f in files:
        try:
            raw = read_any(f)
        except Exception as e:
            print(f"--- {f.relative_to(root)}: unreadable ({e})"); continue
        inspect_frame(raw, str(f.relative_to(root)))


## 3. Load into one unified schema
Columns: `text, label_raw, label_binary, language, source, split` (IndicGuard also has `category`).

* `label_binary`: 0 = Non-Hate, 1 = Hate/Abusive, NaN = unmapped/undecided.
* HateXplain mapping (deliberate): **normal -> 0; offensive and hatespeech -> 1**.
* IndicGuard `label_binary` = 1 if `violated_categories` contains a category from `INDICGUARD_HATE_CATEGORIES`, 0 if the prompt is safe, NaN otherwise (section 9).

* MACD: only `data/raw/macd/dataset/` is read; **raw label 0 = abusive -> label_binary 1** (see 3.2).
* IndicGuard: one row per prompt id (repeated ids and empty/REDACTED prompts are dropped).


In [ ]:
macd = load_macd()
hx   = load_hatexplain()
ig   = load_indicguard()
all_df = pd.concat([macd, hx, ig], ignore_index=True)
print("\nCombined rows:", len(all_df))
all_df.head()

### 3.1 Verify the label mapping (raw label -> binary)
Check that every raw label landed in the right bucket.

In [ ]:
for name, d in [("MACD", macd), ("HateXplain", hx), ("IndicGuard", ig)]:
    if len(d):
        print(f"--- {name}: label_raw vs label_binary ---")
        print(pd.crosstab(d["label_raw"].astype(str), d["label_binary"].fillna(-1).map({0.0:"0 (Non-Hate)",1.0:"1 (Hate/Abusive)",-1.0:"NaN (unmapped)"})))
        print()

### 3.2 Eyeball check: what does each MACD raw label really mean?
**MACD uses `label 0 = abusive` and `label 1 = non-abusive`** (confirmed: label 0 sums to 74,550 over `dataset/`, exactly the paper's "abusive samples"). The loader flips this so that `label_binary 1 = Hate/Abusive`.

Never trust a number you have not looked at: the cell below prints random comments for each raw label. **Read the Tamil ones yourself** - every `raw label = 0` group should be mostly insults/slurs and every `raw label = 1` group mostly ordinary comments.

In [ ]:
for lang in ["Tamil", "Hindi", "Malayalam"]:
    sub = macd[(macd["language"] == lang) & (macd["split"] == "train")]
    for raw_label, g in sub.groupby("label_raw"):
        print(f"\n=== MACD {lang} | raw label = {raw_label}  ->  label_binary = {g['label_binary'].iloc[0]:.0f} "
              f"({'Hate/Abusive' if g['label_binary'].iloc[0] == 1 else 'Non-Hate'}) ===")
        for t in g["text"].sample(6, random_state=1):
            print("  -", str(t)[:110])

## 4. Dataset sizes

In [ ]:
lines = []
for lang in ["Hindi", "Tamil", "Malayalam"]:
    lines.append((f"MACD {lang}", int(((macd["language"] == lang)).sum())))
other_macd = macd[~macd["language"].isin(["Hindi", "Tamil", "Malayalam"])]
for lang, n in other_macd["language"].value_counts().items():
    lines.append((f"MACD {lang}", int(n)))
lines.append(("HateXplain", len(hx)))
lines.append(("IndicGuard", len(ig)))
size_df = pd.DataFrame(lines, columns=["dataset", "rows"])
for ds, n in lines:
    print(f"{ds:<18}: {n}")
size_df

## 5. Class distribution (Non-Hate vs Hate/Abusive)

In [ ]:
labelled = all_df[all_df["source"].isin(["MACD", "HateXplain"]) & all_df["label_binary"].notna()].copy()
labelled["class"] = labelled["label_binary"].map(LABEL_NAMES)
class_dist = (labelled.groupby(["language", "class"]).size().unstack(fill_value=0)
              .reindex(columns=["Non-Hate", "Hate/Abusive"], fill_value=0))
class_dist["total"] = class_dist.sum(axis=1)
class_dist["hate_%"] = (100 * class_dist["Hate/Abusive"] / class_dist["total"]).round(1)
print(class_dist.to_string())
ax = class_dist[["Non-Hate", "Hate/Abusive"]].plot(kind="bar", figsize=(8, 4), rot=0)
ax.set_title("Class distribution by language (MACD + HateXplain)"); ax.set_ylabel("samples")
plt.tight_layout(); plt.savefig(FIGURES_DIR / "class_distribution.png", dpi=150); plt.show()

### 5.1 Class distribution by split

In [ ]:
print(pd.crosstab([labelled["source"], labelled["language"], labelled["split"]], labelled["class"]).to_string())

## 6. Language distribution

In [ ]:
lang_dist = all_df.groupby(["source", "language"]).size().rename("rows").reset_index()
print(lang_dist.to_string(index=False))
plt.figure(figsize=(9, 4))
sns.barplot(data=lang_dist, x="language", y="rows", hue="source")
plt.title("Rows per language and source"); plt.xticks(rotation=30)
plt.tight_layout(); plt.savefig(FIGURES_DIR / "language_distribution.png", dpi=150); plt.show()

## 7. Text length (characters and words)

In [ ]:
all_df["n_chars"] = all_df["text"].astype("string").str.len()
all_df["n_words"] = all_df["text"].astype("string").str.split().str.len()
def length_stats(g):
    return pd.Series({"avg_chars": g.n_chars.mean(), "min_chars": g.n_chars.min(), "max_chars": g.n_chars.max(),
                      "avg_words": g.n_words.mean(), "min_words": g.n_words.min(), "max_words": g.n_words.max()})
len_stats = all_df.groupby(["source", "language"]).apply(length_stats, include_groups=False).round(1)
print(len_stats.to_string())
print("\nOverall:", all_df["n_chars"].agg(["mean", "min", "max"]).round(1).to_dict())
plt.figure(figsize=(9, 4))
sns.boxplot(data=all_df[all_df["n_words"].notna()], x="language", y="n_words", hue="source", showfliers=False)
plt.title("Words per text"); plt.xticks(rotation=30)
plt.tight_layout(); plt.savefig(FIGURES_DIR / "text_length_words.png", dpi=150); plt.show()

## 8. Missing values and duplicates

In [ ]:
rows = []
for name, d in [("MACD", macd), ("HateXplain", hx), ("IndicGuard", ig)]:
    if len(d) == 0:
        continue
    t = d["text"].astype("string")
    rows.append({"dataset": name, "rows": len(d),
                 "missing_text": int(t.isna().sum() + (t.str.strip() == "").sum()),
                 "missing_label": int(d["label_binary"].isna().sum()),
                 "duplicate_text": int(t.duplicated().sum()),
                 "duplicate_text_+_label": int(d.duplicated(subset=["text", "label_raw"]).sum())})
quality = pd.DataFrame(rows); print(quality.to_string(index=False))
# Same text with conflicting labels is a data-quality red flag:
conf = (labelled.groupby("text")["label_binary"].nunique() > 1).sum()
print("\nTexts with conflicting labels:", int(conf))

## 9. IndicGuard: which records are hate-related?

IndicGuard is an LLM *safety* dataset (AEGIS 2.0 taxonomy), so we look before mapping.
Its `violated_categories` column is a comma-separated list per row; this section splits it and shows every individual category.
The default in `src/utils/config.py` treats **`Hate/Identity Hate`** as hate. Confirm the names below; if you want a broader definition (Harassment, Profanity, Threat) edit `INDICGUARD_HATE_CATEGORIES`, restart the kernel and re-run.

In [ ]:
if len(ig) == 0:
    print("IndicGuard not loaded yet.")
else:
    print("Columns:", list(ig.columns))
    for col in ["label_raw", "language", "split"]:
        if col in ig.columns:
            print(f"\n--- value counts: {col} ---")
            print(ig[col].astype("string").value_counts(dropna=False).head(60).to_string())

    # `category` holds comma-separated lists -> split into one row per category
    cats = (ig["category"].astype("string").str.split(",").explode().str.strip())
    cats = cats[cats.notna() & (cats != "")]
    print("\n--- individual categories (after splitting on commas) ---")
    print(cats.value_counts().to_string())

    kw = cats.str.contains("hate|abus|discrim|harass|toxic|slur|offens|profan|threat", case=False, na=False)
    print("\nHate-like category names found:", sorted(cats[kw].unique().tolist()))

    # How the current mapping in config.py behaves:
    print("\n--- label_binary produced by the CURRENT config mapping ---")
    print(ig["label_binary"].map({0.0: "0 Non-Hate", 1.0: "1 Hate/Abusive"}).fillna("NaN (not used)")
            .value_counts().to_string())
    print("\n--- Hate/Abusive rows per language ---")
    print(pd.crosstab(ig["language"], ig["label_binary"].map({0.0: "Non-Hate", 1.0: "Hate/Abusive"}).fillna("unused")).to_string())

    hate_rows = ig[ig["label_binary"] == 1]
    print("\nSample hate-labelled prompts (look at them with your own eyes!):")
    if len(hate_rows):
        print(hate_rows.sample(min(5, len(hate_rows)), random_state=42)[["text", "category", "language"]].to_string(max_colwidth=100))


## 10. Save summaries (small CSVs you can commit)

In [ ]:
size_df.to_csv(METRICS_DIR / "dataset_sizes.csv", index=False)
class_dist.to_csv(METRICS_DIR / "class_distribution.csv")
lang_dist.to_csv(METRICS_DIR / "language_distribution.csv", index=False)
len_stats.to_csv(METRICS_DIR / "text_length_stats.csv")
quality.to_csv(METRICS_DIR / "data_quality.csv", index=False)
print("Saved to", METRICS_DIR)